In [1]:
# Cell 1: Imports
import json
import numpy as np
import pandas as pd
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from contextualized_topic_models.models.ctm import CombinedTM
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from data.dataset import WikitextDataset, BillDataset
from src.config import local_config


In [2]:
local_config

{'data': {'wikitext': 'data/wikitext/train.metadata.jsonl',
  'bill': 'data/bill/train.metadata.jsonl'},
 'output': {'topic_models'},
 'embeddings': {'wikitext': 'data/wikitext/train.metadata.embeddings.jsonl.all-MiniLM-L6-v2.parquet',
  'bill': 'data/bill/train.metadata.embeddings.jsonl.all-MiniLM-L6-v2.parquet'}}

In [3]:
# Cell 2: Load data (small sample for testing)
dataset = BillDataset(data_file=os.path.join("..", local_config['data']['bill']))

# Take only first 100 documents for quick testing
sample_docs = [doc for i, doc in enumerate(dataset) if i < 100]

print(f"Loaded {len(sample_docs)} documents")
print(f"First doc keys: {sample_docs[0].keys()}")
print(f"First doc text sample: {sample_docs[0]['text'][:200]}")

Loaded 100 documents
First doc keys: dict_keys(['id', 'text', 'label', 'sub_categories', 'tokenized_text'])
First doc text sample: Medicare Prescription Drug Price Negotiation Act of 2007 - Amends title XVIII (Medicare) of the Social Security Act to require the Secretary of Health and Human Services to negotiate with pharmaceutic


In [5]:
# Cell 2: Load pre-computed embeddings with PyArrow
import pyarrow.parquet as pq

embeddings_path = os.path.join('..', local_config['embeddings']['bill'])

print(f"Loading from: {embeddings_path}")

# Load parquet file with PyArrow
table = pq.read_table(embeddings_path)

print(f"Loaded embeddings")
print(f"Schema: {table.schema}")
print(f"Number of rows: {len(table)}")
print(f"Column names: {table.column_names}")

Loading from: ../data/bill/train.metadata.embeddings.jsonl.all-MiniLM-L6-v2.parquet
Loaded embeddings
Schema: id: string
summary: string
topic: string
subtopic: string
subjects_top_term: string
date: timestamp[ns]
tokenized_text: string
embeddings: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1018
Number of rows: 32661
Column names: ['id', 'summary', 'topic', 'subtopic', 'subjects_top_term', 'date', 'tokenized_text', 'embeddings']


In [8]:
table

pyarrow.Table
id: string
summary: string
topic: string
subtopic: string
subjects_top_term: string
date: timestamp[ns]
tokenized_text: string
embeddings: string
----
id: [["110-HR-4","110-HR-118","110-HR-205","110-HR-426","110-HR-464",...,"114-HR-2852","114-HR-3360","114-HR-4866","114-HR-5714","114-S-2030"]]
summary: [["Medicare Prescription Drug Price Negotiation Act of 2007 - Amends title XVIII (Medicare) of the So (... 380 chars omitted)","Amends part D (Voluntary Prescription Drug Benefit Program) of title XVIII (Medicare) of the Socia (... 265 chars omitted)","Medicare Prescription Drug Enrollment Extension Act of 2007 - Amends title XVIII (Medicare) of the (... 135 chars omitted)","Medicaid Obesity Treatment Act of 2007 - Amends title XIX (Medicaid) of the Social Security Act to (... 77 chars omitted)","Compassionate Assistance for Rape Emergencies Act of 2007 - Prohibits any federal funds from being (... 613 chars omitted)",...,"FLORICH Act or the Founding Legacies of Reserve Int

In [9]:
df_embeddings

,id,summary,topic,subtopic,subjects_top_term,date,tokenized_text,embeddings
0,110-HR-4,Medicare Prescription Drug Price Negotiation A...,Health,Drug Coverage and Cost,Health,2007-01-05,medicare prescription drug price negotiation a...,-0.054190855 0.07438477 0.04872954 -0.05391701...
1,110-HR-118,Amends part D (Voluntary Prescription Drug Ben...,Health,Drug Coverage and Cost,Health,2007-01-04,amends voluntary prescription drug benefit pro...,-0.07286065 0.117670394 0.06422381 -0.12456543...
2,110-HR-205,Medicare Prescription Drug Enrollment Extensio...,Health,Drug Coverage and Cost,Health,2007-01-04,amends title xviii medicare social security ac...,-0.040297806 0.027768554 0.06305037 -0.0400666...
3,110-HR-426,Medicaid Obesity Treatment Act of 2007 - Amend...,Health,Drug Coverage and Cost,Health,2007-01-11,treatment act amends title xix medicaid social...,-0.025736725 0.049286507 0.067163266 0.0128805...
4,110-HR-464,Compassionate Assistance for Rape Emergencies ...,Health,Drug Coverage and Cost,Health,2007-01-12,compassionate assistance rape emergencies act ...,-0.06253695 0.074354336 -0.034669135 0.0064562...
...,...,...,...,...,...,...,...,...
32656,114-HR-2852,FLORICH Act or the Founding Legacies of Reserv...,Education,Reserve Forces,Armed forces and national security,2015-06-23,act founding reserve integral combat training ...,-0.012931034 0.032478184 -0.068133324 -0.01318...
32657,114-HR-3360,Defense Against Digital Theft Act \n\nThis bil...,Government Operations,Right to Privacy,Government operations and politics,2015-07-29,bill directs provide individuals affected brea...,-0.081255764 0.035028107 0.036376037 0.0253327...
32658,114-HR-4866,Flood Insurance Rate Increase Suspension Act o...,Labor,Disaster Relief,Finance and financial sector,2016-03-23,bill bars taking effect month period increases...,-0.06254896 0.0037069132 0.08670666 -0.0155094...
32659,114-HR-5714,(This measure has not been amended since it wa...,Civil Rights,Postal Service,Government operations and politics,2016-07-11,measure amended introduced summary expanded ac...,-0.039871644 0.026031023 0.04764961 -0.0384587...


In [7]:
# Cell 3: Extract embeddings from the table

# Convert table to pandas
df_embeddings = table.to_pandas()

print(f"DataFrame shape: {df_embeddings.shape}")
print(f"\nFirst row embeddings (raw string):")
print(df_embeddings['embeddings'].iloc[0][:100])  # Show first 100 chars

# Parse the embeddings - they're space-separated floats, not Python lists
print("\nParsing embeddings...")
embeddings_list = df_embeddings['embeddings'].apply(lambda x: [float(num) for num in x.split()]).tolist()
embeddings = np.array(embeddings_list)

print(f"\n✓ Embeddings extracted")
print(f"Embeddings shape: {embeddings.shape}")
print(f"Expected: ({len(df_embeddings)}, 384) for all-MiniLM-L6-v2")
print(f"\nFirst embedding sample (first 5 dimensions):")
print(embeddings[0][:5])

DataFrame shape: (32661, 8)

First row embeddings (raw string):
-0.054190855 0.07438477 0.04872954 -0.053917017 -0.066089794 0.055935644 -0.014089064 0.05068079 0.0

Parsing embeddings...

✓ Embeddings extracted
Embeddings shape: (32661, 384)
Expected: (32661, 384) for all-MiniLM-L6-v2

First embedding sample (first 5 dimensions):
[-0.05419086  0.07438477  0.04872954 -0.05391702 -0.06608979]


In [11]:
# Cell 4: Load all training documents to match embeddings
dataset = BillDataset(data_file=os.path.join("..", local_config['data']['bill']))

# Load ALL documents (not just sample)
all_docs = [doc for doc in dataset]

print(f"✓ Loaded {len(all_docs)} documents")
print(f"✓ Embeddings count: {len(embeddings)}")

# Verify they match
if len(all_docs) != len(embeddings):
    print(f"⚠️  WARNING: Mismatch! {len(all_docs)} docs vs {len(embeddings)} embeddings")
else:
    print(f"✅ Document count matches embeddings!")

print(f"\nFirst doc keys: {all_docs[0].keys()}")
print(f"First doc text sample: {all_docs[0]['text'][:200]}")

✓ Loaded 32661 documents
✓ Embeddings count: 32661
✅ Document count matches embeddings!

First doc keys: dict_keys(['id', 'text', 'label', 'sub_categories', 'tokenized_text'])
First doc text sample: Medicare Prescription Drug Price Negotiation Act of 2007 - Amends title XVIII (Medicare) of the Social Security Act to require the Secretary of Health and Human Services to negotiate with pharmaceutic


In [12]:
# Cell 5: Prepare preprocessed texts for BoW
# Use tokenized_text from all documents
preprocessed_texts = [' '.join(doc['tokenized_text']) for doc in all_docs]

print(f"✓ Total documents: {len(preprocessed_texts)}")
print(f"\nFirst preprocessed text sample (first 200 chars):")
print(preprocessed_texts[0][:200])
print(f"\nPreprocessed text lengths:")
print(f"  Min: {min(len(text.split()) for text in preprocessed_texts)} tokens")
print(f"  Max: {max(len(text.split()) for text in preprocessed_texts)} tokens")
print(f"  Mean: {np.mean([len(text.split()) for text in preprocessed_texts]):.1f} tokens")

✓ Total documents: 32661

First preprocessed text sample (first 200 chars):
medicare prescription drug price negotiation act amends title xviii medicare social security act require secretary health_and_human_services negotiate pharmaceutical manufacturers prices charged presc

Preprocessed text lengths:
  Min: 10 tokens
  Max: 10576 tokens
  Mean: 104.7 tokens


In [14]:
# Cell 6: Load pre-existing vocabulary
with open('../data/bill/vocab.json', 'r') as f:
    vocab_dict = json.load(f)  # {"word": id, ...}

# Create id2word mapping (list indexed by vocab id)
vocab = [None] * len(vocab_dict)
for word, idx in vocab_dict.items():
    vocab[idx] = word

print(f"✓ Vocabulary loaded")
print(f"  Size: {len(vocab)}")
print(f"  First 10 words: {vocab[:10]}")
print(f"  Last 10 words: {vocab[-10:]}")

✓ Vocabulary loaded
  Size: 15000
  First 10 words: ['-century', '-operated', '-registered', '-service', '-service_community_schools_advisory_committee', '1040ez', '1040sr', '10th', '110th', '11th']
  Last 10 words: ['zika', 'zimbabwe', 'zinc', 'zip', 'zone', 'zoned', 'zones', 'zoning', 'zoological', 'zoonotic']


In [17]:
# Cell 7: Create Bag-of-Words representation with pre-existing vocabulary
from sklearn.feature_extraction.text import CountVectorizer

print("Creating BoW representation...")

# Create CountVectorizer with the pre-existing vocabulary
vectorizer = CountVectorizer(
    vocabulary=vocab_dict,  # Use your vocab {word: id}
    lowercase=False,        # Already preprocessed
    token_pattern=r'(?u)\b\w+\b'  # Match word tokens
)

# Transform preprocessed texts to BoW matrix
bow_matrix = vectorizer.fit_transform(preprocessed_texts)

print(f"\n✓ BoW matrix created")
print(f"  Shape: {bow_matrix.shape}")
print(f"  Expected: (32661, 15000)")
print(f"  Type: {type(bow_matrix)}")
print(f"  Sparsity: {(1 - bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])) * 100:.2f}%")

# Show first document's word counts
print(f"\n  First document word counts (non-zero only):")
first_doc = bow_matrix[0].toarray()[0]
non_zero_indices = np.where(first_doc > 0)[0][:10]  # Show first 10
for idx in non_zero_indices:
    print(f"    {vocab[idx]}: {first_doc[idx]}")

Creating BoW representation...

✓ BoW matrix created
  Shape: (32661, 15000)
  Expected: (32661, 15000)
  Type: <class 'scipy.sparse._csr.csr_matrix'>
  Sparsity: 99.58%

  First document word counts (non-zero only):
    act: 2
    amends: 1
    charged: 1
    covered: 1
    drug: 4
    drugs: 1
    eligible: 1
    enrolled: 1
    health_and_human_services: 1
    individuals: 1


In [19]:
# Cell 8: Create custom CTM training dataset with YOUR vocab
from contextualized_topic_models.utils.data_preparation import CTMDataset

print("Creating CTM dataset with pre-computed components...")

# Create the CTM dataset manually with YOUR exact vocab and BoW
training_dataset = CTMDataset(
    X_contextual=embeddings,        # Pre-computed embeddings (32661, 384)
    X_bow=bow_matrix,               # Pre-computed BoW (32661, 15000)
    idx2token=vocab                 # YOUR vocabulary (15000 words)
)

print(f"\n✓ CTM dataset created")
print(f"  Vocab size: {len(vocab)}")
print(f"  BoW shape: {training_dataset.X_bow.shape}")
print(f"  Embeddings shape: {training_dataset.X_contextual.shape}")

# Verify alignment
assert training_dataset.X_bow.shape[1] == len(vocab), "BoW vocab size mismatch!"
assert training_dataset.X_contextual.shape[0] == training_dataset.X_bow.shape[0], "Document count mismatch!"
print(f"✅ All shapes aligned correctly!")

Creating CTM dataset with pre-computed components...

✓ CTM dataset created
  Vocab size: 15000
  BoW shape: (32661, 15000)
  Embeddings shape: (32661, 384)
✅ All shapes aligned correctly!


In [20]:
# Cell 9: Train CTM model
num_topics = 25

print(f"Training CTM with {num_topics} topics...")
print(f"  BoW size: {len(vocab)}")
print(f"  Contextual size: {embeddings.shape[1]}")

ctm = CombinedTM(
    bow_size=len(vocab),              # 15000 (your vocab size)
    contextual_size=embeddings.shape[1],  # 384 (all-MiniLM-L6-v2)
    n_components=num_topics,          # 25 topics
    num_epochs=20,                    # Increase for better results
    batch_size=64,
    lr=2e-3
)

print("\nStarting training...")
ctm.fit(training_dataset)

print(f"\n✅ CTM model trained with {num_topics} topics")

Training CTM with 25 topics...
  BoW size: 15000
  Contextual size: 384

Starting training...


Epoch: [20/20]	 Seen Samples: [652800/653220]	Train Loss: 883.4480491488588	Time: 0:00:59.090147: : 20it [20:02, 60.14s/it]
100%|██████████| 511/511 [00:48<00:00, 10.46it/s] 


✅ CTM model trained with 25 topics


In [27]:
# Cell 9: Train CTM model with 50 epochs
num_topics = 25

print(f"Training CTM with {num_topics} topics...")
print(f"  BoW size: {len(vocab)}")
print(f"  Contextual size: {embeddings.shape[1]}")

ctm = CombinedTM(
    bow_size=len(vocab),              # 15000 (your vocab size)
    contextual_size=embeddings.shape[1],  # 384 (all-MiniLM-L6-v2)
    n_components=num_topics,          # 25 topics
    num_epochs=50,                    # Changed from 20 to 50
    batch_size=64,
    lr=5e-3
)

print("\nStarting training (50 epochs)...")
ctm.fit(training_dataset)

print(f"\n✅ CTM model trained with {num_topics} topics")
print(f"Final loss: {ctm.best_loss_train:.2f}")

Training CTM with 25 topics...
  BoW size: 15000
  Contextual size: 384

Starting training (50 epochs)...


Epoch: [50/50]	 Seen Samples: [1632000/1633050]	Train Loss: 878.8662834616268	Time: 0:01:02.010188: : 50it [50:26, 60.53s/it]
100%|██████████| 511/511 [00:48<00:00, 10.45it/s] 


✅ CTM model trained with 25 topics
Final loss: inf


In [ ]:
# 18 epoch 883.38
# 22 epoch 882.61
# 27 epoch 880
# 29 epochs 882 
# 36 epoch 881
# 40 epochs 881
# 47 epochs 878.64
# 48 epoch 879.74
# 50 epoch 878.866



In [28]:
# Cell 10: Extract and save CTM outputs in standard format
print("=" * 60)
print("CTM MODEL OUTPUTS")
print("=" * 60)

print(f"\n✅ Model trained successfully!")
print(f"Final loss: {ctm.best_loss_train:.2f}")
print(f"Number of topics: {ctm.n_components}")
print(f"Vocabulary size: {len(vocab)}")

# 1. Get topic-word distribution (beta matrix)
topic2word = ctm.get_topic_word_matrix()  # Shape: (25, 15000)
print(f"\nTopic-word matrix shape: {topic2word.shape}")

# 2. Get document-topic distribution (theta matrix)
doc2topic = ctm.get_doc_topic_distribution(training_dataset)  # Shape: (32661, 25)
print(f"Document-topic matrix shape: {doc2topic.shape}")

# 3. Calculate topic-doc distribution (transpose and normalize)
topic_sum = doc2topic.sum(axis=0, keepdims=True)  # Sum across documents for each topic
topic2doc = doc2topic / topic_sum  # Normalize
topic2doc = topic2doc.T  # Transpose to (25, 32661)
print(f"Topic-doc matrix shape: {topic2doc.shape}")

# 4. Save in standard format
output_dir = 'output/topic_models'
os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, f'ctm_bill_{ctm.n_components}.json')

with open(save_path, 'w') as fp:
    json.dump({
        'topic2word_dist': topic2word.tolist(),
        'doc2topic_dist': doc2topic.tolist(),
        'topic2doc_dist': topic2doc.tolist(),
        'vocab': vocab
    }, fp)

print(f"\n✅ Model outputs saved to: {save_path}")
print(f"\nSaved fields:")
print(f"  - topic2word_dist: (25, 15000)")
print(f"  - doc2topic_dist: (32661, 25)")
print(f"  - topic2doc_dist: (25, 32661)")
print(f"  - vocab: 15000 words")

CTM MODEL OUTPUTS

✅ Model trained successfully!
Final loss: inf
Number of topics: 25
Vocabulary size: 15000

Topic-word matrix shape: (25, 15000)


100%|██████████| 511/511 [00:55<00:00,  9.24it/s] 


Document-topic matrix shape: (32661, 25)
Topic-doc matrix shape: (25, 32661)

✅ Model outputs saved to: output/topic_models/ctm_bill_25.json

Saved fields:
  - topic2word_dist: (25, 15000)
  - doc2topic_dist: (32661, 25)
  - topic2doc_dist: (25, 32661)
  - vocab: 15000 words


In [48]:
doc2topic.sum(axis=1).sum()  # Each document's topic distribution sums to 1

32661.0

In [33]:
topic2word.sum(axis=1).shape  # Each topic's word distribution sums to 1

(25,)

In [35]:
topic2word[0]

array([0.8872431 , 0.889721  , 0.88694507, ..., 1.4931672 , 1.1925039 ,
       1.2126522 ], dtype=float32)

In [39]:
topic_2_word_distribution = ctm.get_topic_word_distribution()
topic_2_word_distribution.sum(axis=1)

array([1.        , 0.99999994, 0.99999994, 0.99999994, 1.        ,
       1.        , 1.        , 1.        , 0.99999994, 1.        ,
       0.99999994, 1.        , 0.99999994, 1.        , 1.0000001 ,
       0.99999994, 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 0.99999994, 1.        , 1.        ],
      dtype=float32)

In [44]:
topic_2_word_distribution.shape

(25, 15000)